# Resemble Enhance (Denoise Only)

In [1]:
import warnings
from pathlib import Path
from tqdm import tqdm
import torch
import torchaudio
from clear_memory import clear_memory

warnings.filterwarnings('ignore')

/opt/anaconda3/envs/denoise_resemble/lib/python3.10/site-packages/torch/cuda/__init__.py:51: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
input_dir = Path('../ad_detection/data/raw/Pitt')
output_dir = Path('../ad_detection/data/denoised/Pitt-Resemble')

control_files = list((input_dir / 'Control').glob('*.wav'))
dementia_files = list((input_dir / 'Dementia').glob('*.wav'))

## Load Model

In [3]:
from resemble_enhance.enhancer.inference import denoise as resemble_denoise

# 设备检测
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print(f"Device: {device}")
print("Model will be auto-downloaded on first call (from HuggingFace)")

[2026-03-16 15:30:43,843] [INFO] [real_accelerator.py:161:get_accelerator] Setting ds_accelerator to mps (auto detect)


[2026-03-16 15:30:44,267] torch.distributed.elastic.multiprocessing.redirects: [WARNING] NOTE: Redirects are currently not supported in Windows or MacOs.


Device: mps
Model will be auto-downloaded on first call (from HuggingFace)


## Denoise Function

In [4]:
def denoise_audio(audio_path, device):
    """
    使用 Resemble Enhance 进行语音降噪（denoise only）
    """
    dwav, sr = torchaudio.load(str(audio_path))

    # 多声道转单声道
    dwav = dwav.mean(0)  # (channels, time) -> (time,)

    # resemble_denoise 内部会自动重采样到 44100Hz 并处理
    hwav, out_sr = resemble_denoise(dwav=dwav, sr=sr, device=device, run_dir=None)

    return hwav, out_sr

In [5]:
def batch_denoise(files, output_subdir, device, group_name):
    """
    批量降噪处理（每个文件前后都清理显存）

    Args:
        files: 待处理的音频文件列表
        output_subdir: 输出子目录
        device: 计算设备
        group_name: 组名（用于显示进度）
    """
    output_subdir.mkdir(parents=True, exist_ok=True)

    success_count = 0
    skip_count = 0
    fail_count = 0

    for audio_file in tqdm(files, desc=f"Processing {group_name}"):
        output_file = output_subdir / audio_file.name

        # 跳过已处理的文件
        if output_file.exists():
            skip_count += 1
            continue

        try:
            clear_memory()

            # 降噪
            denoised_audio, sr = denoise_audio(audio_file, device)

            # 保存（16位整数格式）
            torchaudio.save(str(output_file), denoised_audio[None], sr)
            success_count += 1

            del denoised_audio
            clear_memory()

        except Exception as e:
            fail_count += 1
            print(f"\nFailed: {audio_file.name}: {e}")
            clear_memory()

    print(f"\n{group_name} 处理完成:")
    print(f"成功: {success_count}")
    print(f"跳过: {skip_count}")
    print(f"失败: {fail_count}")
    print(f"总计: {len(files)}")

## Execute denoise function

In [6]:
clear_memory()

batch_denoise(
    dementia_files,
    output_dir / 'Dementia',
    device,
    group_name='Dementia'
)

clear_memory()

batch_denoise(
    control_files,
    output_dir / 'Control',
    device,
    group_name='Control'
)

Processing Dementia:   0%|          | 0/309 [00:00<?, ?it/s]Cloning into '/opt/anaconda3/envs/denoise_resemble/lib/python3.10/site-packages/resemble_enhance/model_repo'...
git-lfs filter-process: git-lfs: command not found
fatal: the remote end hung up unexpectedly
You can inspect what was checked out with 'git status'
and retry with 'git restore --source=HEAD :/'

Processing Dementia:   0%|          | 1/309 [00:00<02:21,  2.17it/s]


Failed: 007-3.wav: Failed to clone the repository, please try again.


git: 'lfs' is not a git command. See 'git --help'.

The most similar command is
	refs
Processing Dementia:   1%|          | 2/309 [00:00<02:02,  2.50it/s]

Already up to date.

Failed: 023-2.wav: Failed to pull latest changes, please try again.


git: 'lfs' is not a git command. See 'git --help'.

The most similar command is
	refs
Processing Dementia:   1%|          | 3/309 [00:01<01:51,  2.75it/s]

...

git: 'lfs' is not a git command. See 'git --help'.

The most similar command is
	refs
Processing Dementia:  20%|██        | 63/309 [00:21<01:50,  2.23it/s]

Already up to date.

Failed: 164-1.wav: Failed to pull latest changes, please try again.


git: 'lfs' is not a git command. See 'git --help'.

The most similar command is
	refs
Processing Dementia:  20%|██        | 63/309 [00:22<01:26,  2.84it/s]

Already up to date.

Failed: 319-0.wav: Failed to pull latest changes, please try again.


KeyboardInterrupt: 